In [9]:
!python -V

Python 3.9.6


In [10]:
import pandas as pd

In [11]:
import pickle

In [12]:
import seaborn as sns
import matplotlib.pyplot as plt

In [27]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import root_mean_squared_error

In [14]:
import mlflow


mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='mlflow-artifacts:/950814995213236933', creation_time=1749559785763, experiment_id='950814995213236933', last_update_time=1749559785763, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [20]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [21]:
df_train = read_dataframe('./data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('./data/green_tripdata_2021-02.parquet')


In [22]:
len(df_train), len(df_val)

(73908, 61921)

In [23]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [24]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [25]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [28]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

/Users/anastasijatrizna/Desktop/mlops-zoomcamp/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:311: RuntimeWarning: divide by zero encountered in matmul
  intercept_ = y_offset - X_offset @ coef_
/Users/anastasijatrizna/Desktop/mlops-zoomcamp/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:311: RuntimeWarning: overflow encountered in matmul
  intercept_ = y_offset - X_offset @ coef_
/Users/anastasijatrizna/Desktop/mlops-zoomcamp/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:311: RuntimeWarning: invalid value encountered in matmul
  intercept_ = y_offset - X_offset @ coef_


7.758715204796038

In [29]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [31]:
with mlflow.start_run():

    mlflow.set_tag("developer", "cristian")

    mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.csv")
    mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.csv")

    alpha = 0.1
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")

🏃 View run awesome-snail-228 at: http://127.0.0.1:5000/#/experiments/950814995213236933/runs/b7332976220448e19cb6d39ba2dc9e7a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/950814995213236933


/Users/anastasijatrizna/Desktop/mlops-zoomcamp/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:311: RuntimeWarning: divide by zero encountered in matmul
  intercept_ = y_offset - X_offset @ coef_
/Users/anastasijatrizna/Desktop/mlops-zoomcamp/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:311: RuntimeWarning: overflow encountered in matmul
  intercept_ = y_offset - X_offset @ coef_
/Users/anastasijatrizna/Desktop/mlops-zoomcamp/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:311: RuntimeWarning: invalid value encountered in matmul
  intercept_ = y_offset - X_offset @ coef_


In [ ]:
#brew install libomp to solve 32 on 64 error
import xgboost as xgb

In [37]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [38]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [42]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [43]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

[0]	validation-rmse:6.88643                           
[1]	validation-rmse:6.67535                           
[2]	validation-rmse:6.65919                           
  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

/Users/anastasijatrizna/Desktop/mlops-zoomcamp/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:50:16] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[3]	validation-rmse:6.64840                           
[4]	validation-rmse:6.63896                           
[5]	validation-rmse:6.62992                           
[6]	validation-rmse:6.62043                           
[7]	validation-rmse:6.61752                           
[8]	validation-rmse:6.61501                           
[9]	validation-rmse:6.60927                           
[10]	validation-rmse:6.60549                          
[11]	validation-rmse:6.60101                          
[12]	validation-rmse:6.59446                          
[13]	validation-rmse:6.59099                          
[14]	validation-rmse:6.58730                          
[15]	validation-rmse:6.58275                          
[16]	validation-rmse:6.56963                          
[17]	validation-rmse:6.56857                          
[18]	validation-rmse:6.56662                          
[19]	validation-rmse:6.56181                          
[20]	validation-rmse:6.55756                          
[21]	valid

/Users/anastasijatrizna/Desktop/mlops-zoomcamp/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:50:21] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:11.18516                                                    
[1]	validation-rmse:10.32833                                                    
[2]	validation-rmse:9.62417                                                     
[3]	validation-rmse:9.04418                                                     
[4]	validation-rmse:8.57275                                                     
[5]	validation-rmse:8.18982                                                     
[6]	validation-rmse:7.87902                                                     
[7]	validation-rmse:7.62978                                                     
[8]	validation-rmse:7.43009                                                     
[9]	validation-rmse:7.26854                                                     
[10]	validation-rmse:7.13761                                                    
[11]	validation-rmse:7.03659                                                    
[12]	validation-rmse:6.95044

...

/Users/anastasijatrizna/Desktop/mlops-zoomcamp/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:59:59] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[2]	validation-rmse:6.68239                                                     
[3]	validation-rmse:6.66625                                                     
[4]	validation-rmse:6.65445                                                     
[5]	validation-rmse:6.65205                                                     
[6]	validation-rmse:6.65065                                                     
[7]	validation-rmse:6.64789                                                     
[8]	validation-rmse:6.64513                                                     
[9]	validation-rmse:6.64442                                                     
[10]	validation-rmse:6.64152                                                    
[11]	validation-rmse:6.64207                                                    
[12]	validation-rmse:6.64071                                                    
[13]	validation-rmse:6.63917                                                    
[14]	validation-rmse:6.63717

/Users/anastasijatrizna/Desktop/mlops-zoomcamp/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [11:00:08] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:11.14871                                                    
[1]	validation-rmse:10.26603                                                    
[2]	validation-rmse:9.53832                                                     
[3]	validation-rmse:8.94432                                                     
[4]	validation-rmse:8.46239                                                     
[5]	validation-rmse:8.07351                                                     
[6]	validation-rmse:7.76082                                                     
[7]	validation-rmse:7.51088                                                     
[8]	validation-rmse:7.31156                                                     
🏃 View run victorious-duck-358 at: http://127.0.0.1:5000/#/experiments/950814995213236933/runs/d972c7fab55143e5b146a0688c2ddda0

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/950814995213236933   

 38%|███▊      | 19/50 [09:54<16:09, 31.28s/trial, best loss:

KeyboardInterrupt: 

In [44]:
mlflow.xgboost.autolog(disable=True)

In [45]:
with mlflow.start_run():
    
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.09585355369315604,
        'max_depth': 30,
        'min_child_weight': 1.060597050922164,
        'objective': 'reg:linear',
        'reg_alpha': 0.018060244040060163,
        'reg_lambda': 0.011658731377413597,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

[0]	validation-rmse:11.44482


/Users/anastasijatrizna/Desktop/mlops-zoomcamp/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [11:00:19] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)


[1]	validation-rmse:10.77202
[2]	validation-rmse:10.18363
[3]	validation-rmse:9.67396
[4]	validation-rmse:9.23166
[5]	validation-rmse:8.84808
[6]	validation-rmse:8.51883
[7]	validation-rmse:8.23597
[8]	validation-rmse:7.99320
[9]	validation-rmse:7.78709
[10]	validation-rmse:7.61022
[11]	validation-rmse:7.45952
[12]	validation-rmse:7.33049
[13]	validation-rmse:7.22098
[14]	validation-rmse:7.12713
[15]	validation-rmse:7.04752
[16]	validation-rmse:6.98005
[17]	validation-rmse:6.92232
[18]	validation-rmse:6.87112
[19]	validation-rmse:6.82740
[20]	validation-rmse:6.78995
[21]	validation-rmse:6.75792
[22]	validation-rmse:6.72994
[23]	validation-rmse:6.70547
[24]	validation-rmse:6.68390
[25]	validation-rmse:6.66421
[26]	validation-rmse:6.64806
[27]	validation-rmse:6.63280
[28]	validation-rmse:6.61924
[29]	validation-rmse:6.60773
[30]	validation-rmse:6.59777
[31]	validation-rmse:6.58875
[32]	validation-rmse:6.58107
[33]	validation-rmse:6.57217
[34]	validation-rmse:6.56557
[35]	validation-rmse:

/Users/anastasijatrizna/Desktop/mlops-zoomcamp/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [11:00:49] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2025/06/11 11:00:51 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run resilient-turtle-998 at: http://127.0.0.1:5000/#/experiments/950814995213236933/runs/f305c2c4fe3c42ed9497034c30dc1f0d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/950814995213236933


In [46]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import LinearSVR

mlflow.sklearn.autolog()

for model_class in (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, LinearSVR):

    with mlflow.start_run():

        mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.csv")
        mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.csv")
        mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

        mlmodel = model_class()
        mlmodel.fit(X_train, y_train)

        y_pred = mlmodel.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)
        

🏃 View run unequaled-goat-573 at: http://127.0.0.1:5000/#/experiments/950814995213236933/runs/4f73dbbc434a4156b237e60a1081e59d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/950814995213236933


KeyboardInterrupt: 